# Exercise 3.1 — Overlap of Haar-Random States and Measure Concentration

**Chapter 3: Haar Ensembles** &nbsp;|&nbsp; *Section 3.1: Haar-Random States*

---

## Motivation

How much do two randomly chosen quantum states overlap?  In low dimensions, there is significant variability.  But in high dimensions, a remarkable phenomenon occurs: the overlap $|\langle\psi|\phi\rangle|^2$ **concentrates** around $1/D$ — two random states are nearly orthogonal with overwhelming probability.  This concentration of measure underlies the typicality arguments used throughout quantum information: random states behave like typical states, and typical states are nearly maximally entangled.

## Exercise Statement

The squared overlap $\chi = |\langle\psi|\phi\rangle|^2$ between two independent Haar-random pure states in $D$ dimensions has density:

$$
P(\chi) = (D-1)(1-\chi)^{D-2}, \qquad \chi \in [0, 1].
$$

(This is a $\mathrm{Beta}(1, D-1)$ distribution.)

**(a)** Compute $\mathbb{E}[\chi]$ and verify $\mathbb{E}[\chi] = 1/D$.

**(b)** Compute $\mathrm{Var}(\chi)$ and show it scales as $1/D^2$ for large $D$.

## Solution

### Part (a): Mean overlap

Using the substitution $u = 1 - \chi$, we have $\chi = 1 - u$, $d\chi = -du$:

$$
\mathbb{E}[\chi] = (D-1)\int_0^1 \chi(1-\chi)^{D-2}\,d\chi = (D-1)\int_0^1 (1-u)\, u^{D-2}\, du.
$$

Expanding:

$$
= (D-1)\left[\frac{1}{D-1} - \frac{1}{D}\right] = 1 - \frac{D-1}{D} = \frac{1}{D}.
$$

$$
\boxed{\mathbb{E}[\chi] = \frac{1}{D}.}
$$

Two random states have expected overlap $1/D$ — they are nearly orthogonal in high dimensions.

### Part (b): Variance

First compute the second moment:

$$
\mathbb{E}[\chi^2] = (D-1)\int_0^1 (1-u)^2\, u^{D-2}\, du = (D-1)\left[\frac{1}{D-1} - \frac{2}{D} + \frac{1}{D+1}\right].
$$

Simplifying:

$$
\mathbb{E}[\chi^2] = 1 - \frac{2(D-1)}{D} + \frac{D-1}{D+1} = \frac{2}{D(D+1)}.
$$

The variance is:

$$
\mathrm{Var}(\chi) = \frac{2}{D(D+1)} - \frac{1}{D^2} = \frac{2D - (D+1)}{D^2(D+1)} = \frac{D-1}{D^2(D+1)}.
$$

$$
\boxed{\mathrm{Var}(\chi) = \frac{D-1}{D^2(D+1)} \sim \frac{1}{D^2} \quad \text{as } D \to \infty.}
$$

The standard deviation scales as $1/D$, which is the **same order** as the mean itself.  This means the *relative* fluctuation $\sigma/\mu \sim 1/\sqrt{D} \to 0$: the overlap concentrates around $1/D$ with vanishing relative spread.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

D = sp.Symbol('D', positive=True, integer=True)

# Haar overlap: E[|<psi|phi>|^2] = 1/D  (first moment)
E_overlap = sp.Rational(1, 1) / D
print(f'E[|<psi|phi>|^2] = {E_overlap}')

# Second moment: E[|<psi|phi>|^4] = 2/(D(D+1))
E_overlap4 = sp.Rational(2, 1) / (D * (D + 1))
print(f'E[|<psi|phi>|^4] = {E_overlap4}')

# Variance
Var = sp.simplify(E_overlap4 - E_overlap**2)
print(f'Var[|<psi|phi>|^2] = {Var}')
expected = (D - 1) / (D**2 * (D + 1))
assert sp.simplify(Var - expected) == 0
print(f'Verified: (D-1)/[D^2(D+1)]')

# For large D, overlaps ~ Exp(D): concentrated near 0
print(f'\nTypical overlap for N qubits:')
for n in range(2, 12):
    d = 2**n
    print(f'  N={n:2d}: E[overlap] = {1/d:.6f}')

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)

print(f"{'D':>5s}  {'E[χ]':>8s}  {'1/D':>8s}  {'Var(χ)':>10s}  {'pred':>10s}  {'σ/μ':>8s}")
print(f"{'─'*5:>5s}  {'─'*8:>8s}  {'─'*8:>8s}  {'─'*10:>10s}  {'─'*10:>10s}  {'─'*8:>8s}")

for D in [2, 4, 8, 16, 32]:
    n_samples = 20000
    overlaps = []
    for _ in range(n_samples):
        psi = unitary_group.rvs(D)[:, 0]
        phi = unitary_group.rvs(D)[:, 0]
        overlaps.append(abs(psi.conj() @ phi)**2)
    
    E_chi = np.mean(overlaps)
    V_chi = np.var(overlaps)
    V_pred = (D-1) / (D**2*(D+1))
    rel = np.sqrt(V_chi) / E_chi
    
    print(f"{D:5d}  {E_chi:8.4f}  {1/D:8.4f}  {V_chi:10.6f}  {V_pred:10.6f}  {rel:8.4f}")
    assert abs(E_chi - 1/D) < 0.01

print("\nConcentration: σ/μ → 0 as D → ∞. ✓")